In [1]:
import numpy as np
import tensorflow as tf
from tensorflow import keras
import pickle
from pathlib import Path
import os

# STEP 1: Load Trained Model and Data
print("\n" + "="*60)
print("STEP 1: Loading Model and Data")
print("="*60)

model_path = Path('../models/activity_model.h5')
model = keras.models.load_model(model_path)
print(f"Loaded model from: {model_path}")

with open('../data/processed/preprocessing_info.pkl', 'rb') as f:
    preprocessing_info = pickle.load(f)

X_test = np.load('../data/processed/X_test.npy')
y_test = np.load('../data/processed/y_test.npy')
X_train = np.load('../data/processed/X_train.npy')

print(f"Train data: {X_train.shape}")
print(f"Test data: {X_test.shape}")

# STEP 2: Evaluate Original Model
print("\n" + "="*60)
print("STEP 2: Original Model Performance")
print("="*60)

loss_original, acc_original = model.evaluate(X_test, y_test, verbose=0)
print(f"Original Model:")
print(f"  Accuracy: {acc_original*100:.2f}%")
print(f"  Loss: {loss_original:.4f}")

model_size_original = os.path.getsize(model_path) / 1024
print(f"  File size: {model_size_original:.2f} KB")

# STEP 3: Convert to TFLite INT8
print("\n" + "="*60)
print("STEP 3: Converting to TensorFlow Lite (INT8)")
print("="*60)

def representative_dataset_gen():
    # Use ALL training data for best calibration
    for i in range(len(X_train)):
        sample = X_train[i:i+1].astype(np.float32)
        yield [sample]

print(f"Using ALL {len(X_train)} training samples for calibration")

converter = tf.lite.TFLiteConverter.from_keras_model(model)
converter.optimizations = [tf.lite.Optimize.DEFAULT]
converter.representative_dataset = representative_dataset_gen
converter.target_spec.supported_ops = [tf.lite.OpsSet.TFLITE_BUILTINS_INT8]
converter.inference_input_type = tf.int8
converter.inference_output_type = tf.int8

print("Converting model...")
tflite_model = converter.convert()

tflite_path = Path('../models/activity_model_quantized.tflite')
with open(tflite_path, 'wb') as f:
    f.write(tflite_model)

tflite_size = len(tflite_model) / 1024
size_reduction = ((model_size_original - tflite_size) / model_size_original) * 100
print(f"Saved: {tflite_path}")
print(f"  File size: {tflite_size:.2f} KB (reduction: {size_reduction:.1f}%)")

# STEP 4: Test Quantized Model Accuracy
print("\n" + "="*60)
print("STEP 4: Testing Quantized Model")
print("="*60)

interpreter = tf.lite.Interpreter(model_path=str(tflite_path))
interpreter.allocate_tensors()

input_details = interpreter.get_input_details()
output_details = interpreter.get_output_details()

print(f"Input shape: {input_details[0]['shape']}, type: {input_details[0]['dtype']}")
print(f"Output shape: {output_details[0]['shape']}, type: {output_details[0]['dtype']}")

input_scale = input_details[0]['quantization'][0]
input_zero_point = input_details[0]['quantization'][1]
output_scale = output_details[0]['quantization'][0]
output_zero_point = output_details[0]['quantization'][1]

print(f"Input scale: {input_scale:.6f}, zero_point: {input_zero_point}")
print(f"Output scale: {output_scale:.6f}, zero_point: {output_zero_point}")

print("\nRunning inference on test set...")
predictions = []

for i in range(len(X_test)):
    input_data = X_test[i:i+1].astype(np.float32)
    input_data_quantized = (input_data / input_scale + input_zero_point).astype(np.int8)
    
    interpreter.set_tensor(input_details[0]['index'], input_data_quantized)
    interpreter.invoke()
    
    output_data = interpreter.get_tensor(output_details[0]['index'])
    output_dequantized = (output_data.astype(np.float32) - output_zero_point) * output_scale
    
    predictions.append(np.argmax(output_dequantized))

predictions = np.array(predictions)
acc_quantized = np.mean(predictions == y_test)
acc_drop = (acc_original - acc_quantized) * 100

print(f"\nQuantized Model:")
print(f"  Accuracy: {acc_quantized*100:.2f}%")
print(f"  Accuracy drop: {acc_drop:.2f}%")

# STEP 5: Comparison Summary
print("\n" + "="*60)
print("STEP 5: Comparison Summary")
print("="*60)

print(f"{'Metric':<25} {'Original':<20} {'Quantized':<20}")
print("-"*65)
print(f"{'Format':<25} {'Keras (.h5)':<20} {'TFLite INT8':<20}")
print(f"{'File Size':<25} {f'{model_size_original:.1f} KB':<20} {f'{tflite_size:.1f} KB':<20}")
print(f"{'Test Accuracy':<25} {f'{acc_original*100:.2f}%':<20} {f'{acc_quantized*100:.2f}%':<20}")
print(f"{'Accuracy Drop':<25} {'-':<20} {f'{acc_drop:.2f}%':<20}")

# STEP 6: Convert to C Header File
print("\n" + "="*60)
print("STEP 6: Converting to C Header File")
print("="*60)

with open(tflite_path, 'rb') as f:
    tflite_bytes = f.read()

def create_c_array(data, var_name):
    c_array = f"// Auto-generated file\n"
    c_array += f"// TensorFlow Lite model for activity recognition\n"
    c_array += f"// Model size: {len(data)} bytes\n\n"
    c_array += f"alignas(8) const unsigned char {var_name}[] = {{\n"
    for i in range(0, len(data), 12):
        row = data[i:i+12]
        c_array += "  " + ", ".join([f"0x{b:02x}" for b in row])
        if i + 12 < len(data):
            c_array += ","
        c_array += "\n"
    c_array += f"}};\n"
    c_array += f"const unsigned int {var_name}_len = {len(data)};\n"
    return c_array

c_header = create_c_array(tflite_bytes, "activity_model_data")

c_header_path = Path('../models/model_data.h')
with open(c_header_path, 'w') as f:
    f.write(c_header)

print(f"Saved: {c_header_path}")
print(f"  Array size: {len(tflite_bytes)} bytes")

# STEP 7: Create Config Header for C++
print("\n" + "="*60)
print("STEP 7: Creating C++ Config Header")
print("="*60)

num_classes = len(preprocessing_info['label_to_activity'])
num_features = len(preprocessing_info['mean'])

if 'config' in preprocessing_info and isinstance(preprocessing_info['config'], dict):
    window_size = preprocessing_info['config'].get('window_size', 128)
    sampling_rate = preprocessing_info['config'].get('sampling_rate', 50)
else:
    window_size = 128
    sampling_rate = 50

print(f"  NUM_CLASSES: {num_classes}")
print(f"  NUM_FEATURES: {num_features}")
print(f"  WINDOW_SIZE: {window_size}")
print(f"  SAMPLING_RATE: {sampling_rate}")

config_content = f"""// Auto-generated configuration file
// Generated from Python preprocessing

#ifndef CONFIG_H
#define CONFIG_H

// Model configuration
#define NUM_CLASSES {num_classes}
#define NUM_FEATURES {num_features}
#define WINDOW_SIZE {window_size}
#define SAMPLING_RATE {sampling_rate}

// Normalization parameters (from training data)
const float SENSOR_MEAN[NUM_FEATURES] = {{
    {', '.join([f'{m:.6f}f' for m in preprocessing_info['mean']])}
}};

const float SENSOR_STD[NUM_FEATURES] = {{
    {', '.join([f'{s:.6f}f' for s in preprocessing_info['std']])}
}};

// Activity labels
const char* ACTIVITY_LABELS[NUM_CLASSES] = {{
    {', '.join([f'"{preprocessing_info["label_to_activity"][i]}"' for i in range(num_classes)])}
}};

// Sensor column order (for reference)
// 0: Ax, 1: Ay, 2: Az, 3: Gx, 4: Gy, 5: Gz

#endif // CONFIG_H
"""

config_path = Path('../models/config.h')
with open(config_path, 'w') as f:
    f.write(config_content)

print(f"Saved: {config_path}")

print("\n" + "="*60)
print("QUANTIZATION COMPLETE!")
print("="*60)
print(f"  Original: {acc_original*100:.2f}% accuracy, {model_size_original:.1f} KB")
print(f"  Quantized: {acc_quantized*100:.2f}% accuracy, {tflite_size:.1f} KB")
print(f"  Size reduction: {size_reduction:.1f}%, Accuracy drop: {acc_drop:.2f}%")
print(f"\nFiles: model_data.h, config.h, activity_model_quantized.tflite")
print(f"Status: {'READY FOR DEPLOYMENT' if acc_drop < 5 else 'Check accuracy drop'}")


STEP 1: Loading Model and Data


2026-05-28 23:52:38.872919: I metal_plugin/src/device/metal_device.cc:1154] Metal device set to: Apple M1
2026-05-28 23:52:38.872962: I metal_plugin/src/device/metal_device.cc:296] systemMemory: 8.00 GB
2026-05-28 23:52:38.872981: I metal_plugin/src/device/metal_device.cc:313] maxCacheSize: 2.67 GB
2026-05-28 23:52:38.873065: I tensorflow/core/common_runtime/pluggable_device/pluggable_device_factory.cc:303] Could not identify NUMA node of platform GPU ID 0, defaulting to 0. Your kernel may not have been built with NUMA support.
2026-05-28 23:52:38.873118: I tensorflow/core/common_runtime/pluggable_device/pluggable_device_factory.cc:269] Created TensorFlow device (/job:localhost/replica:0/task:0/device:GPU:0 with 0 MB memory) -> physical PluggableDevice (device: 0, name: METAL, pci bus id: <undefined>)


Loaded model from: ../models/activity_model.h5
Train data: (8700, 128, 6)
Test data: (2052, 128, 6)

STEP 2: Original Model Performance


2026-05-28 23:52:39.397451: I tensorflow/core/grappler/optimizers/custom_graph_optimizer_registry.cc:114] Plugin optimizer for device_type GPU is enabled.


Original Model:
  Accuracy: 84.16%
  Loss: 0.7585
  File size: 368.25 KB

STEP 3: Converting to TensorFlow Lite (INT8)
Using ALL 8700 training samples for calibration
Converting model...
INFO:tensorflow:Assets written to: /var/folders/52/bpf46hx95sl2yr7m61mcmnlw0000gn/T/tmpsgni8k4r/assets


INFO:tensorflow:Assets written to: /var/folders/52/bpf46hx95sl2yr7m61mcmnlw0000gn/T/tmpsgni8k4r/assets
/opt/homebrew/Caskroom/miniforge/base/envs/tinyml2/lib/python3.10/site-packages/tensorflow/lite/python/convert.py:887: UserWarning: Statistics for quantized inputs were expected, but not specified; continuing anyway.
  warnings.warn(
2026-05-28 23:52:41.006088: W tensorflow/compiler/mlir/lite/python/tf_tfl_flatbuffer_helpers.cc:364] Ignored output_format.
2026-05-28 23:52:41.006112: W tensorflow/compiler/mlir/lite/python/tf_tfl_flatbuffer_helpers.cc:367] Ignored drop_control_dependency.
2026-05-28 23:52:41.006562: I tensorflow/cc/saved_model/reader.cc:45] Reading SavedModel from: /var/folders/52/bpf46hx95sl2yr7m61mcmnlw0000gn/T/tmpsgni8k4r
2026-05-28 23:52:41.007980: I tensorflow/cc/saved_model/reader.cc:91] Reading meta graph with tags { serve }
2026-05-28 23:52:41.007985: I tensorflow/cc/saved_model/reader.cc:132] Reading SavedModel debug info (if present) from: /var/folders/52/bpf4

Saved: ../models/activity_model_quantized.tflite
  File size: 39.09 KB (reduction: 89.4%)

STEP 4: Testing Quantized Model
Input shape: [  1 128   6], type: <class 'numpy.int8'>
Output shape: [ 1 11], type: <class 'numpy.int8'>
Input scale: 0.170063, zero_point: -34
Output scale: 0.003906, zero_point: -128

Running inference on test set...

Quantized Model:
  Accuracy: 53.31%
  Accuracy drop: 30.85%

STEP 5: Comparison Summary
Metric                    Original             Quantized           
-----------------------------------------------------------------
Format                    Keras (.h5)          TFLite INT8         
File Size                 368.2 KB             39.1 KB             
Test Accuracy             84.16%               53.31%              
Accuracy Drop             -                    30.85%              

STEP 6: Converting to C Header File
Saved: ../models/model_data.h
  Array size: 40024 bytes

STEP 7: Creating C++ Config Header
  NUM_CLASSES: 11
  NUM_FEATURES: 

fully_quantize: 0, inference_type: 6, input_inference_type: INT8, output_inference_type: INT8
INFO: Created TensorFlow Lite XNNPACK delegate for CPU.
